# Credit Card Fraud Detection

This notebook builds a binary classifier for fraudulent (`Class=1`) versus legitimate (`Class=0`) card transactions using a public Kaggle dataset and a modest TensorFlow/Keras neural network. It inspects the data, preserves a held-out test set, fits preprocessing on training data only, chooses a threshold using validation data, and evaluates test data at the end.

> **Run from the project root.** The notebook downloads the current public dataset CSV through KaggleHub and stages it at `data/creditcard.csv`. No dataset or model outputs are included.

The dataset is extremely imbalanced. Accuracy alone is not adequate for fraud detection; draw conclusions from the metrics produced by running this notebook, not assumed scores.

## 1. Download and stage the dataset

The first data cell requests the exact `creditcard.csv` file from Kaggle's public `mlg-ulb/creditcardfraud` dataset through KaggleHub, then copies the returned file into the project's expected `data/creditcard.csv` path. Internet access is required. Kaggle authentication may be needed if Kaggle requires user consent for this public resource; never add credentials to the notebook or project. `kaggle kernels pull ...` retrieves optional notebook/source reference files only and is not needed to obtain the dataset.

In [ ]:
from pathlib import Path
import shutil

import kagglehub

DATASET_HANDLE = 'mlg-ulb/creditcardfraud'
DATA_PATH = Path('data/creditcard.csv')
KAGGLEHUB_PATH = kagglehub.dataset_download(DATASET_HANDLE, path='creditcard.csv')
downloaded_csv_path = Path(KAGGLEHUB_PATH)

if not downloaded_csv_path.exists():
    raise FileNotFoundError(
        f"KaggleHub returned a path that does not exist: {KAGGLEHUB_PATH!r}. "
        'Check internet access and try running this cell again.'
    )
if not downloaded_csv_path.is_file():
    raise FileNotFoundError(
        f"KaggleHub returned a path that is not a file: {KAGGLEHUB_PATH!r}. "
        'Expected the dataset file creditcard.csv.'
    )

DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
if downloaded_csv_path.resolve() != DATA_PATH.resolve():
    shutil.copy2(downloaded_csv_path, DATA_PATH)
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"The dataset copy was not created at {DATA_PATH.resolve()}")

print(f'KaggleHub returned path: {KAGGLEHUB_PATH}')
print(f'Local project dataset: {DATA_PATH.resolve()}')

## 2. Imports and reproducibility

A fixed seed makes data partitions and initialization reproducible within the limits of the installed libraries and hardware. Different TensorFlow versions or devices can still produce small numerical differences.

In [ ]:
import os

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

tf.keras.utils.set_random_seed(SEED)
sns.set_theme(style='whitegrid')
print(f'TensorFlow version: {tf.__version__}')

## 3. Load and validate the CSV

The canonical file is expected to contain `Time`, `V1`–`V28`, and `Amount` as numeric inputs, plus the binary `Class` target. Report the observed dimensions and inspect data types, missing values, duplicate rows, and valid target values. Class counts are examined after splitting, using the training subset only.

In [ ]:
df = pd.read_csv(DATA_PATH)
feature_columns = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount']
expected_columns = set(feature_columns + ['Class'])
missing_columns = sorted(expected_columns.difference(df.columns))
if missing_columns:
    raise ValueError(f'The CSV is missing expected columns: {missing_columns}')

non_numeric = sorted(set(feature_columns).difference(df[feature_columns].select_dtypes(include=np.number).columns))
if non_numeric:
    raise TypeError(f'Expected numeric input features, found non-numeric columns: {non_numeric}')
if df['Class'].isna().any() or not set(df['Class'].unique()).issubset({0, 1}):
    raise ValueError('Class must be non-null and contain only binary labels 0 and 1.')

print(f'Rows: {len(df):,} | Columns: {df.shape[1]}')
print(f'Input feature count: {len(feature_columns)}')
print(f'Target values: {sorted(df["Class"].unique().tolist())}')
display(df.head())
display(df.dtypes.rename('dtype').to_frame())
display(df.isna().sum().rename('missing_values').to_frame())
print(f'Rows containing any missing value: {int(df.isna().any(axis=1).sum()):,}')
print(f'Exact duplicate rows: {int(df.duplicated().sum()):,}')
if df[feature_columns].isna().any().any():
    raise ValueError('Input features contain missing values; inspect and resolve them before modeling.')

## 4. Stratified train / validation / test split

Partition before label-based exploratory analysis. Stratify using the target so each partition retains the rare-class proportion. Hold out 20% for the final test set first. From the remaining 80%, reserve 18.75% for validation, which gives approximately 65% train, 15% validation, and 20% test overall. All class-distribution, class-conditioned plot, and feature-versus-`Class` analyses below use only the training subset; validation and test labels are not used for EDA. Test labels are not used for preprocessing, training, early stopping, or threshold selection; they are used only in the final evaluation section.

In [ ]:
X = df[feature_columns].copy()
y = df['Class'].astype('int32')

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1875, random_state=SEED + 1, stratify=y_train_val
)

split_rows = [
    {'partition': 'Train', 'rows': len(X_train)},
    {'partition': 'Validation', 'rows': len(X_val)},
    {'partition': 'Test', 'rows': len(X_test)},
]
display(pd.DataFrame(split_rows).set_index('partition'))
print(f'Partition proportions: train={len(X_train) / len(X):.3f}, validation={len(X_val) / len(X):.3f}, test={len(X_test) / len(X):.3f}')

## 5. Exploratory data analysis (training subset only)

All label-based EDA in this section uses only the training subset: class counts and distribution, class-conditioned Amount and Time plots, and feature-versus-`Class` correlations. Validation and test labels are excluded from these analyses. When legitimate transactions dominate, an always-legitimate classifier may look accurate while missing every fraud. `Time` is measured in seconds from the first transaction in this dataset, not as a calendar timestamp. These plots describe the training subset and do not establish future performance.

In [ ]:
train_eda = X_train.copy()
train_eda['Class'] = y_train
class_counts = y_train.value_counts().reindex([0, 1], fill_value=0)
class_distribution = pd.DataFrame({
    'transaction_count': class_counts,
    'percentage': 100 * class_counts / len(y_train),
})
display(class_distribution.rename(index={0: 'Legitimate (0)', 1: 'Fraud (1)'}))

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
sns.barplot(x=['Legitimate (0)', 'Fraud (1)'], y=class_counts.values, ax=axes[0], color='#4472C4')
axes[0].set(title='Training split class balance', xlabel='Class', ylabel='Transactions')
axes[0].tick_params(axis='x', rotation=15)

plot_data = train_eda[['Amount', 'Time', 'Class']].copy()
plot_data['Log amount (log1p)'] = np.log1p(plot_data['Amount'])
plot_data['Time (hours from start)'] = plot_data['Time'] / 3600
sns.histplot(data=plot_data, x='Log amount (log1p)', hue='Class', stat='density',
             common_norm=False, element='step', fill=False, bins=50, ax=axes[1])
axes[1].set(title='Transaction amount by class (training split)', xlabel='log(1 + Amount)', ylabel='Class-conditional density')
sns.histplot(data=plot_data, x='Time (hours from start)', hue='Class', stat='density',
             common_norm=False, element='step', fill=False, bins=50, ax=axes[2])
axes[2].set(title='Transaction time by class (training split)', xlabel='Hours from dataset start', ylabel='Class-conditional density')
plt.tight_layout()
plt.show()

In [ ]:
# Training-only correlation is descriptive, not a causal explanation of anonymized features.
class_correlations = (
    train_eda[feature_columns + ['Class']]
    .corr(numeric_only=True)['Class']
    .drop('Class')
    .abs()
    .sort_values(ascending=False)
    .head(10)
    .sort_values()
)
display(class_correlations.rename('absolute_correlation_with_Class').to_frame())
ax = class_correlations.plot(kind='barh', figsize=(8, 5), color='#70AD47')
ax.set(title='Top 10 absolute feature correlations with Class (training split)', xlabel='Absolute Pearson correlation', ylabel='Feature')
plt.tight_layout()
plt.show()

## 6. Scale numeric inputs without leakage

A dense neural network benefits from features on comparable scales. Fit `StandardScaler` using only training inputs, then apply that fitted transformation to validation and test inputs. Fitting a scaler on all rows would allow held-out distribution information to affect training.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
print(f'Scaled matrix shapes — train: {X_train_scaled.shape}, validation: {X_val_scaled.shape}')

## 7. Build and train the neural network

The Sequential model is intentionally modest: two ReLU hidden layers (64 and 32 units), Dropout after each layer, and one sigmoid output for fraud probability. Binary cross-entropy is the standard binary-classification loss, and Adam is a practical adaptive optimizer.

Compute `class_weight` from training labels only so errors on the rare fraud class contribute more to the loss. Weighting may improve attention to the minority class, but it does not create new information or guarantee calibrated probabilities. Early stopping monitors validation PR-AUC (`val_pr_auc`) because precision-recall behavior is informative under severe imbalance; restoring the best observed weights avoids keeping a later, weaker epoch.

In [ ]:
classes = np.array([0, 1], dtype=np.int32)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train.to_numpy())
class_weight = {int(label): float(weight) for label, weight in zip(classes, class_weights)}
print(f'Training-only class weights: {class_weight}')

model = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.AUC(curve='PR', name='pr_auc'),
        tf.keras.metrics.AUC(curve='ROC', name='roc_auc'),
    ],
)
model.summary()

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_pr_auc', mode='max', patience=5,
    restore_best_weights=True, verbose=1,
)
history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    class_weight=class_weight,
    epochs=50, batch_size=2048,
    callbacks=[early_stopping], verbose=1,
)

In [ ]:
history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history_df['loss'], label='Train')
axes[0].plot(history_df['val_loss'], label='Validation')
axes[0].set(title='Binary cross-entropy', xlabel='Epoch', ylabel='Loss')
axes[0].legend()
axes[1].plot(history_df['pr_auc'], label='Train PR-AUC')
axes[1].plot(history_df['val_pr_auc'], label='Validation PR-AUC')
axes[1].set(title='Precision-recall AUC', xlabel='Epoch', ylabel='PR-AUC')
axes[1].legend()
plt.tight_layout()
plt.show()
print(f'Epochs completed: {len(history_df)}')

## 8. Choose a decision threshold using validation data only

A sigmoid model produces probabilities; converting them into labels requires a threshold. This demonstration chooses the validation threshold that maximizes F1. It is not a universal optimum: F1 treats precision and recall symmetrically, does not encode monetary or operational costs, and selecting a threshold on one validation set can overfit that set. Do not use test labels to change the threshold.

In [ ]:
validation_probabilities = model.predict(X_val_scaled, verbose=0).ravel()
precision_values, recall_values, threshold_values = precision_recall_curve(y_val, validation_probabilities)
f1_candidates = (
    2 * precision_values[:-1] * recall_values[:-1]
    / np.maximum(precision_values[:-1] + recall_values[:-1], np.finfo(float).eps)
)
best_index = int(np.nanargmax(f1_candidates))
selected_threshold = float(threshold_values[best_index])
validation_predictions = (validation_probabilities >= selected_threshold).astype('int32')
print(f'Validation-selected threshold (max F1): {selected_threshold:.6f}')
print(f'Validation precision: {precision_score(y_val, validation_predictions, zero_division=0):.4f}')
print(f'Validation recall:    {recall_score(y_val, validation_predictions, zero_division=0):.4f}')
print(f'Validation F1:        {f1_score(y_val, validation_predictions, zero_division=0):.4f}')

## 9. Final held-out test evaluation

This is the one final test evaluation. Reuse the validation-selected threshold as-is. Average Precision summarizes the precision-recall curve and is the primary threshold-free summary here; ROC-AUC is included as a secondary ranking metric. Precision, recall, F1, the confusion matrix, and the classification report describe behavior at the selected threshold. Do not revise the model or threshold after seeing the test results.

In [ ]:
# Transform test inputs with the scaler fitted on training data; do not fit or tune anything on test data.
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Predict on the held-out test partition once; all test metrics reuse these predictions.
test_probabilities = model.predict(X_test_scaled, verbose=0).ravel()
test_predictions = (test_probabilities >= selected_threshold).astype('int32')

test_average_precision = average_precision_score(y_test, test_probabilities)
test_roc_auc = roc_auc_score(y_test, test_probabilities)
test_precision = precision_score(y_test, test_predictions, zero_division=0)
test_recall = recall_score(y_test, test_predictions, zero_division=0)
test_f1 = f1_score(y_test, test_predictions, zero_division=0)
test_confusion = confusion_matrix(y_test, test_predictions, labels=[0, 1])

print(f'Decision threshold: {selected_threshold:.6f}')
print(f'Average Precision (PR-AUC summary): {test_average_precision:.4f}')
print(f'ROC-AUC (secondary):               {test_roc_auc:.4f}')
print(f'Precision:                         {test_precision:.4f}')
print(f'Recall:                            {test_recall:.4f}')
print(f'F1:                                {test_f1:.4f}')
print('\nConfusion matrix (rows = actual, columns = predicted; labels 0, 1):')
display(pd.DataFrame(test_confusion, index=['Actual legitimate (0)', 'Actual fraud (1)'],
                     columns=['Predicted legitimate (0)', 'Predicted fraud (1)']))
print('Classification report at the validation-selected threshold:')
print(classification_report(y_test, test_predictions, labels=[0, 1],
                            target_names=['Legitimate (0)', 'Fraud (1)'], zero_division=0))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(test_confusion, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted legitimate', 'Predicted fraud'],
            yticklabels=['Actual legitimate', 'Actual fraud'], ax=ax)
ax.set(title='Held-out test confusion matrix', xlabel='Prediction', ylabel='Actual class')
plt.tight_layout()
plt.show()

# Majority-class baseline: always predict the legitimate class.
majority_baseline_predictions = np.zeros(len(y_test), dtype='int32')
baseline_accuracy = np.mean(majority_baseline_predictions == y_test.to_numpy())
baseline_fraud_recall = recall_score(y_test, majority_baseline_predictions, pos_label=1, zero_division=0)
print(f'Majority-class baseline accuracy: {baseline_accuracy:.4f}')
print(f'Majority-class baseline fraud recall: {baseline_fraud_recall:.4f}')

In [ ]:
print('Run-specific result summary (complete your discussion using these observed outputs):')
print(f'- Held-out test Average Precision: {test_average_precision:.4f}')
print(f'- Held-out test ROC-AUC: {test_roc_auc:.4f}')
print(f'- At threshold {selected_threshold:.6f}: precision={test_precision:.4f}, recall={test_recall:.4f}, F1={test_f1:.4f}')
print(f'- Confusion matrix [TN, FP; FN, TP]: {test_confusion.tolist()}')
print(f'- Always-legitimate baseline: accuracy={baseline_accuracy:.4f}, fraud recall={baseline_fraud_recall:.4f}')

## 10. Discussion and limitations

After running every cell, write a short conclusion here using the run-specific metrics and plots above. Do not use expected or invented scores. Address: (1) how much fraud the chosen threshold detected and how many false alarms it produced, using the confusion matrix; (2) what Average Precision and ROC-AUC add beyond the threshold-specific report; (3) how the majority-class baseline's accuracy compares with its fraud recall; and (4) whether the precision-recall tradeoff is appropriate for the intended use.

Keep these limits in the conclusion: the dataset covers only a short historical window of approximately two days in 2013; anonymized PCA-derived `V1`–`V28` features prevent straightforward semantic explanations; and a random stratified split does not establish generalization to future time periods, changing fraud patterns, or production conditions.